In [74]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [75]:
path = r'C:\Users\Magsihim_AI\Desktop\Project_Statbotics\actions_data\datasets\actions_df.csv'

df = pd.read_csv(path)

In [76]:
df

,score,autoLine,autoReef_botRow_nodeA,autoReef_botRow_nodeB,autoReef_botRow_nodeC,autoReef_botRow_nodeD,autoReef_botRow_nodeE,autoReef_botRow_nodeF,autoReef_botRow_nodeG,autoReef_botRow_nodeH,...,teleopReef_topRow_nodeH,teleopReef_topRow_nodeI,teleopReef_topRow_nodeJ,teleopReef_topRow_nodeK,teleopReef_topRow_nodeL,teleopReef_trough,wallAlgaeCount,team_number,possition,win
0,124,True,False,False,False,False,False,False,False,False,...,True,True,False,False,True,0.0,0.0,2783,1,False
1,144,True,False,False,False,False,False,False,False,False,...,True,True,True,True,False,0.0,0.0,2783,1,False
2,129,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,5.0,2.0,5045,1,True
3,118,True,False,False,False,False,False,False,False,False,...,True,True,True,True,True,1.0,0.0,538,1,True
4,67,True,False,False,False,False,False,False,False,False,...,False,False,True,True,True,1.0,1.0,6107,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114367,64,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,6.0,0.0,9995,3,False
114368,72,True,False,False,False,False,False,True,False,False,...,False,False,False,False,False,1.0,0.0,9991,3,True
114369,77,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,0.0,0.0,9994,3,False
114370,79,True,False,False,False,False,False,False,False,False,...,False,False,True,True,True,14.0,0.0,9993,3,False


In [77]:
player_col = "team_number"
# Identify all non-player columns
non_player_cols = [c for c in df.columns if c != player_col]

# Suppose you already know which are action columns (e.g., prefixed "act_")
# If not, define manually:
action_cols = [c for c in df.columns if ("node" in c.lower())]  # example rule
team_features = [c for c in non_player_cols if c not in action_cols]


In [78]:
le = LabelEncoder()
df["team_encoded"] = le.fit_transform(df[player_col])

# 4️⃣ Define features and labels




In [79]:
X = df[team_features + ["team_encoded"]]
y = df[action_cols]


In [80]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [81]:
print("Action columns:", action_cols)
print("Team features:", team_features)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Sample of X columns:", X.columns.tolist()[:10])
print("Sample of y columns:", y.columns.tolist()[:10])


Action columns: ['autoReef_botRow_nodeA', 'autoReef_botRow_nodeB', 'autoReef_botRow_nodeC', 'autoReef_botRow_nodeD', 'autoReef_botRow_nodeE', 'autoReef_botRow_nodeF', 'autoReef_botRow_nodeG', 'autoReef_botRow_nodeH', 'autoReef_botRow_nodeI', 'autoReef_botRow_nodeJ', 'autoReef_botRow_nodeK', 'autoReef_botRow_nodeL', 'autoReef_midRow_nodeA', 'autoReef_midRow_nodeB', 'autoReef_midRow_nodeC', 'autoReef_midRow_nodeD', 'autoReef_midRow_nodeE', 'autoReef_midRow_nodeF', 'autoReef_midRow_nodeG', 'autoReef_midRow_nodeH', 'autoReef_midRow_nodeI', 'autoReef_midRow_nodeJ', 'autoReef_midRow_nodeK', 'autoReef_midRow_nodeL', 'autoReef_topRow_nodeA', 'autoReef_topRow_nodeB', 'autoReef_topRow_nodeC', 'autoReef_topRow_nodeD', 'autoReef_topRow_nodeE', 'autoReef_topRow_nodeF', 'autoReef_topRow_nodeG', 'autoReef_topRow_nodeH', 'autoReef_topRow_nodeI', 'autoReef_topRow_nodeJ', 'autoReef_topRow_nodeK', 'autoReef_topRow_nodeL', 'teleopReef_botRow_nodeA', 'teleopReef_botRow_nodeB', 'teleopReef_botRow_nodeC', 't

In [82]:
model = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
))
model.fit(X_train, y_train)

MultiOutputRegressor(estimator=RandomForestRegressor(n_jobs=-1,
                                                     random_state=42))

In [83]:
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
print(f"\nRMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")


RMSE: 0.220
R²:   0.378


In [84]:
players = [1323, 1690, 2910]  # replace with actual team_numbers
team_example = X_test.iloc[0].copy() 
X_new = pd.DataFrame([team_example.values] * len(players), columns=team_example.index)
X_new["team_encoded"] = le.transform(players)


In [85]:
pred_actions = pd.DataFrame(model.predict(X_new), columns=action_cols, index=players)


In [86]:
bool_actions = [c for c in action_cols if df[c].dropna().isin([0, 1]).all()]
int_actions  = [c for c in action_cols if c not in bool_actions]

# For each boolean action — keep only the player with the highest probability
for col in bool_actions:
    best_player = pred_actions[col].idxmax()
    pred_actions.loc[players != best_player, col] = 0
    pred_actions.loc[best_player, col] = 1

# Final result
print("\nPredicted Actions (per player):")
pred_actions.round(2)


Predicted Actions (per player):


,autoReef_botRow_nodeA,autoReef_botRow_nodeB,autoReef_botRow_nodeC,autoReef_botRow_nodeD,autoReef_botRow_nodeE,autoReef_botRow_nodeF,autoReef_botRow_nodeG,autoReef_botRow_nodeH,autoReef_botRow_nodeI,autoReef_botRow_nodeJ,...,teleopReef_topRow_nodeC,teleopReef_topRow_nodeD,teleopReef_topRow_nodeE,teleopReef_topRow_nodeF,teleopReef_topRow_nodeG,teleopReef_topRow_nodeH,teleopReef_topRow_nodeI,teleopReef_topRow_nodeJ,teleopReef_topRow_nodeK,teleopReef_topRow_nodeL
1323,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1690,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2910,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0


In [87]:
# df.drop(['win', 'score'], axis = 1, inplace = True)

In [88]:
# # X = player names
# X = df[['team_number']]

# # y = all 81 action columns
# action_cols = df.columns[df.columns != 'team_number']
# y = df[action_cols]

# # Encode player names
# le_player = LabelEncoder()
# X_encoded = le_player.fit_transform(X['team_number']).reshape(-1, 1)


In [89]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_encoded, y, test_size=0.2, random_state=42
# )


In [90]:
# # Random Forest for multi-output
# rf = RandomForestClassifier(n_estimators=100, random_state=42)
# multi_rf = MultiOutputClassifier(rf)
# multi_rf.fit(X_train, y_train)


In [91]:
# df['endGame'].value_counts()

In [92]:
# df['possition'].value_counts()

In [93]:
# # Example: predict actions for player "Romi"
# player_input = le_player.transform([1580]).reshape(-1, 1)
# predicted_actions = multi_rf.predict(player_input)

# # Convert to a readable DataFrame
# predicted_df = pd.DataFrame(predicted_actions, columns=action_cols)
# df_transposed = predicted_df.T.reset_index()
# df_transposed.rename(columns={'value': 'Feature'}, inplace=True)
# df_transposed.loc[(df_transposed[0] == True) | (df_transposed[0] >= 1)]


In [94]:
# df_transposed


In [95]:
# df_transposed.columns

In [96]:
# y_pred = multi_rf.predict(X_test)

# # Example: accuracy per action column
# for i, col in enumerate(action_cols):
#     acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
#     print(f"{col}: {acc:.2f}")
